# 02. Geração fundamentada e placar
Três configurações, com o mesmo modelo local (qwen2.5:3b no Ollama, temperatura 0) e os mesmos 27 casos:
- **A sem contexto**: o modelo responde sozinho.
- **B RAG simples**: busca os 3 trechos mais parecidos e responde só com eles, citando a fonte, ou recusa.
- **C RAG + agente**: o modelo escolhe uma ferramenta (consultar o risco de chuva do ChuvaRota ou buscar na base) e a resposta passa por uma verificação da fonte.

In [1]:
import sys
sys.path.insert(0, '../src')
import json
import pandas as pd
from avaliar import carregar_casos
import configs
from avaliar import rodar, resumo
casos = carregar_casos()
risco = {a['id']: a['risco'] for a in json.load(open('../../mvp/data/alertas.json'))['alertas']}
todos = {nome: rodar(f, casos, risco) for nome, f in configs.CONFIGURACOES.items()}
json.dump(todos, open('../avaliacao/resultados.json', 'w'), ensure_ascii=False, indent=1)
print('ok')

ok


## Placar geral

In [2]:
placar = pd.DataFrame({n: resumo(r) for n, r in todos.items()}).T
placar.to_csv('../avaliacao/placar.csv')
placar

,acertos (%),acerto com fonte certa (%),alucinações (%),JSON válido (%),citação certa (%),recusa correta (%)
A sem contexto,30,0,44,100,0,0
B RAG simples,78,78,0,100,71,100
C RAG + agente,89,89,0,100,86,100


## Por tipo de pergunta
Orientações (18 perguntas com resposta na base), não consta (6 perguntas fora da base) e risco atual (3 perguntas que só a ferramenta do ChuvaRota responde).

In [3]:
linhas = []
for n, r in todos.items():
    for tipo in ('resposta', 'nao_consta', 'ferramenta'):
        rs = [x for x in r if x['tipo'] == tipo]
        linhas.append({'config': n, 'tipo': tipo, 'acertos': f"{sum(x['ok'] for x in rs)}/{len(rs)}", 'alucinações': sum(x['alucinou'] for x in rs)})
pd.DataFrame(linhas).pivot(index='config', columns='tipo', values=['acertos', 'alucinações'])

acertos                     alucinações                    
tipo           ferramenta nao_consta resposta  ferramenta nao_consta resposta
config                                                                       
A sem contexto        0/3        0/6     8/18           3          6        3
B RAG simples         0/3        6/6    15/18           0          0        0
C RAG + agente        3/3        6/6    15/18           0          0        0

## Onde cada configuração errou

In [4]:
for n, r in todos.items():
    print('=====', n)
    for x in r:
        if not x['ok'] or x['alucinou']:
            s = x['saida']
            print(f"caso {x['caso']} ({x['tipo']}) | {x['pergunta']}\n   -> recusa={s.get('recusa')} fonte={s.get('fonte_id')} motivo={s.get('motivo')} | {str(s.get('resposta'))[:150]}")

===== A sem contexto
caso 2 (resposta) | Como descubro se minha casa está em área de risco?
   -> recusa=False fonte=Portal da Prefeitura de São Paulo ou Corpo de Bombeiros motivo=None | Para descobrir se sua casa está em área de risco, você pode consultar o site do Corpo de Bombeiros ou o Portal da Prefeitura de São Paulo, onde inform
caso 3 (resposta) | O que devo fazer com meus documentos se houver risco de alagamento?
   -> recusa=False fonte=None motivo=None | Se houver risco de alagamento, é recomendável guardar seus documentos em um local seguro e elevado, como um cofre ou armário de armazenamento de docum
caso 5 (resposta) | Qual o telefone da Defesa Civil?
   -> recusa=False fonte=None motivo=None | O telefone da Defesa Civil de São Paulo é (11) 3011-8910.
caso 7 (resposta) | Como tratar a água antes de beber depois da enchente?
   -> recusa=False fonte=None motivo=None | Antes de beber água após a enchente, é importante filtrá-la ou deixá-la esfriar para eliminar possíveis im